# Problem 1/2 Metric-Aligned Review Notebook

이 노트북은 김건우 팀원의 지적을 반영한 Problem 1/2 최신 검토본입니다.

핵심 수정:

- Random-unitary diffusion의 x축 `k`는 discrete circuit layer index입니다.
- Hamiltonian projected diffusion의 x축 `t`는 continuous evolution time입니다.
- 따라서 `k`와 `t`를 같은 x축 위치로 직접 비교하면 안 됩니다.
- native parameter curve는 각각 따로 보고, cross-mechanism 비교는 같은 metric 값에 가까운 지점을 matching해서 수행합니다.

이 노트북은 최신 실험 결과를 `results/problem_1_2_baseline/`와 `results/problem_3_seed_sweep/`에서 읽습니다.


In [ ]:
from pathlib import Path
import csv
import re

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path(r"C:\Coding\Hackathon6Quantum")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

P12_DIR = PROJECT_ROOT / "results" / "problem_1_2_baseline"
P3_SUMMARY = PROJECT_ROOT / "results" / "problem_3_seed_sweep" / "seed_sweep_summary.md"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem 1/2 result dir exists:", P12_DIR.exists())
print("Problem 3 seed summary exists:", P3_SUMMARY.exists())


## Optional: regenerate latest Problem 1/2 baseline

이미 최신 결과가 있으면 아래 `RUN_BASELINE`은 `False`로 두면 됩니다. 새 실험 결과를 만들고 싶을 때만 `True`로 바꾸세요.


In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    import subprocess
    subprocess.run(
        ["python", "scripts/run_problem_1_2_baselines.py"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Regenerated Problem 1/2 baseline results.")
else:
    print("Using existing generated results.")


## 1. Native parameter curves

아래 그림은 두 메커니즘을 좌우 subplot으로 분리해서 보여줍니다.

- 왼쪽: random-unitary의 native parameter `step k`
- 오른쪽: Hamiltonian projection의 native parameter `time t`

두 subplot의 가로 위치는 서로 대응하지 않습니다. 즉, `k=1`과 `t=1`이 같은 diffusion strength라는 뜻이 아닙니다.


In [ ]:
def show_png(path, title=None, figsize=(11, 5)):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    image = plt.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

show_png(P12_DIR / "distance_curves.png", "Native parameter curves: k and t are separate axes")


## 2. Metric-aligned comparison

아래 그림은 x축을 `k`나 `t`가 아니라 metric 값으로 바꿔 비교합니다.

- 왼쪽: 각 점을 `(MMD, Wasserstein-type distance)` metric space에 표시합니다.
- 오른쪽: MMD 기준, Wasserstein 기준으로 가장 가까운 non-initial pair를 찾아 비교합니다.

이 방식이 김건우 팀원의 지적에 대한 핵심 수정입니다.


In [ ]:
show_png(P12_DIR / "metric_aligned_comparison.png", "Metric-aligned comparison and nearest comparable-strength pairs")


## 3. Latest Problem 1/2 summary numbers

아래 cell은 최신 `problem_1_2_summary.md`와 CSV를 읽어, 보고서에 들어갈 숫자를 확인합니다.


In [ ]:
def read_csv_rows(path):
    with Path(path).open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

summary_path = P12_DIR / "problem_1_2_summary.md"
summary_text = summary_path.read_text(encoding="utf-8")
print(summary_text)

random_rows = read_csv_rows(P12_DIR / "random_unitary_metrics.csv")
ham_rows = read_csv_rows(P12_DIR / "hamiltonian_metrics.csv")
match_rows = read_csv_rows(P12_DIR / "comparable_strength_resource_matches.csv")

print("\nComparable-strength table")
for row in match_rows:
    print(
        f"matched_by={row['matched_by']}: "
        f"random k={row['random_step']} vs ham t={float(row['hamiltonian_time']):.6f}, "
        f"MMD gap={float(row['mmd_gap']):.6f}, "
        f"W gap={float(row['wasserstein_gap']):.6f}, "
        f"random controls={row['random_controls']}, "
        f"entanglers={row['random_two_qubit_entanglers']}, "
        f"ham fixed terms={row['hamiltonian_fixed_terms']}"
    )


## 4. 직접 x축 비교를 피한 재구성 plot

아래 plot은 같은 cell 안에서 그리지만, `k`와 `t`를 같은 x축으로 섞지 않습니다. 첫 줄은 native parameter curve이고, 둘째 줄은 metric-space comparison입니다.


In [ ]:
def as_float(row, key):
    return float(row[key])

random_k = [as_float(row, "parameter") for row in random_rows]
random_mmd = [as_float(row, "mmd") for row in random_rows]
random_w = [as_float(row, "wasserstein") for row in random_rows]
ham_t = [as_float(row, "parameter") for row in ham_rows]
ham_mmd = [as_float(row, "mmd") for row in ham_rows]
ham_w = [as_float(row, "wasserstein") for row in ham_rows]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

axes[0, 0].plot(random_k, random_mmd, "o-", label="MMD")
axes[0, 0].plot(random_k, random_w, "s--", label="Wasserstein-type")
axes[0, 0].set_title("Random-unitary native curve")
axes[0, 0].set_xlabel("step k")
axes[0, 0].set_ylabel("distance to S0")
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

axes[0, 1].plot(ham_t, ham_mmd, "o-", label="MMD")
axes[0, 1].plot(ham_t, ham_w, "s--", label="Wasserstein-type")
axes[0, 1].set_title("Hamiltonian projection native curve")
axes[0, 1].set_xlabel("time t")
axes[0, 1].set_ylabel("distance to S0")
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend()

axes[1, 0].scatter(random_mmd[1:], random_w[1:], label="random-unitary", alpha=0.8)
axes[1, 0].scatter(ham_mmd[1:], ham_w[1:], label="Hamiltonian projection", alpha=0.8)
axes[1, 0].set_title("Metric-space comparison")
axes[1, 0].set_xlabel("MMD distance")
axes[1, 0].set_ylabel("Wasserstein-type distance")
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend()

labels = [row["matched_by"] for row in match_rows]
random_metric_values = []
ham_metric_values = []
for row in match_rows:
    metric = row["matched_by"]
    if metric == "mmd":
        random_metric_values.append(float(row["random_mmd"]))
        ham_metric_values.append(float(row["hamiltonian_mmd"]))
    else:
        random_metric_values.append(float(row["random_wasserstein"]))
        ham_metric_values.append(float(row["hamiltonian_wasserstein"]))

x = np.arange(len(labels))
width = 0.36
axes[1, 1].bar(x - width / 2, random_metric_values, width, label="random-unitary")
axes[1, 1].bar(x + width / 2, ham_metric_values, width, label="Hamiltonian projection")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_title("Nearest comparable-strength metric values")
axes[1, 1].set_ylabel("matched metric value")
axes[1, 1].grid(axis="y", alpha=0.25)
axes[1, 1].legend()

plt.show()


## 5. Problem 3 latest seed sweep result

Problem 3는 Problem 1/2 baseline 위에 얹는 main extension입니다. 최신 seed sweep 결과도 같은 노트북에 고정해 둡니다.


In [ ]:
def get_md_value(text, label):
    pattern = rf"^\s*(?:-\s*)?{re.escape(label)}\s*:\s*`([^`]+)`"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1) if match else ""

p3_text = P3_SUMMARY.read_text(encoding="utf-8")
print(p3_text)

p3_metrics = {
    "recommendation": get_md_value(p3_text, "Main-claim recommendation"),
    "total_seeds": get_md_value(p3_text, "Total seeds"),
    "use_as_main": get_md_value(p3_text, "use_as_main"),
    "main_candidate_row_fraction": get_md_value(p3_text, "main_candidate row fraction"),
    "median_mmd_improvement": get_md_value(p3_text, "continuous_mmd_improvement"),
    "median_wasserstein_improvement": get_md_value(p3_text, "continuous_wasserstein_improvement"),
    "axis_only_score_margin": get_md_value(p3_text, "continuous_score_minus_axis_score"),
    "diversity_retention": get_md_value(p3_text, "continuous_diversity_retention"),
    "success_probability": get_md_value(p3_text, "continuous_mean_success_probability"),
}
print("\nParsed Problem 3 metrics")
for key, value in p3_metrics.items():
    print(f"{key}: {value}")


## 6. 김건우 팀원 이슈에 대한 답변 초안

지적이 맞습니다. Random-unitary의 `k`와 Hamiltonian projection의 `t`는 서로 다른 물리적/알고리즘적 parameter이므로, 한 그래프에서 같은 x축 위치를 읽어 비교하면 안 됩니다.

수정 방향은 다음과 같습니다.

1. `distance_curves.png`에서는 `k`와 `t`를 좌우 subplot으로 분리합니다.
2. 두 메커니즘을 비교할 때는 `MMD`와 `Wasserstein-type distance`라는 같은 output metric 기준으로 matching합니다.
3. Problem 2(d)의 resource/control 비교는 `comparable_strength_resource_matches.csv`의 matched pair를 기준으로 설명합니다.

따라서 최종 보고서에서는 “`k=1`과 `t=1`을 비교했다”고 쓰면 안 됩니다. 대신 “MMD 기준으로 random step 1과 Hamiltonian `t=0.333333`이 가장 가까운 diffusion strength였고, Wasserstein 기준으로 random step 7과 Hamiltonian `t=3.333333`이 가장 가까웠다”고 쓰는 것이 안전합니다.


## 7. Safe report wording

Problem 1은 random circuit layer 수 `k`에 따른 scrambling baseline이고, Problem 2는 fixed Hamiltonian evolution time `t`와 complement projection에 따른 projected diffusion baseline이다. 두 parameter는 같은 단위가 아니므로 horizontal axis 위치를 직접 비교하지 않는다. 대신 같은 initial ensemble `S0`와 같은 fidelity-based MMD, infidelity-cost Wasserstein metric을 사용해 diffusion strength가 비슷한 지점을 matching하고, 그 지점에서 control/resource proxy를 비교한다.

최신 결과에서 MMD 기준 comparable pair는 random-unitary `k=1`과 Hamiltonian `t=0.333333`이며 MMD gap은 `0.002259`이다. Wasserstein 기준 comparable pair는 random-unitary `k=7`과 Hamiltonian `t=3.333333`이며 Wasserstein gap은 `0.001220`이다. 이 비교는 Hamiltonian 방법이 절대적으로 우월하다는 뜻이 아니라, random layer-wise control과 fixed Hamiltonian time/projection control이 서로 다른 비용 구조를 갖는다는 trade-off를 보여준다.

Problem 3 seed sweep는 `20/20 use_as_main`이며, median MMD improvement `0.097056`, median Wasserstein improvement `0.147983`을 보인다. 다만 axis-only score margin은 `0.010000`으로 작으므로, hardware advantage나 general quantum advantage가 아니라 small-scale post-selected toy proxy에서의 재현 가능한 개선으로 제한해 주장한다.
